# Construcción de la tabla fact resultado

1. Leer la tabla silver `resultados`
1. Leer la tabla silver  `sprints`
1. Agregar nueva colunma calculada `tipo_session` with values `Carrera` o `Sprint`
1.UNION `resultados` y `sprints`
1. Agregar columnas adcionales
    - is_win -> Indica que el piloto ganó la carrera
    - is_podium -> Indica que el piloto llegó en las  posiciones  (1, 2, 3)
    - has_points -> Indica que el piloto gano puntos
1. Escribir la data transformada a la tabla gold `fact_resultado_sesion`



#### 



#### 


In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
target_table = f"{catalogo}.{esquema_gold}.fact_resultado_sesion"

In [0]:
from pyspark.sql import functions as F

#### 1 - Leer las tablas silver fuente
- `silver.resultados`
- `silver.sprints`

In [0]:
results_df = (
    spark.table(f"{catalogo}.{esquema_silver}.results")
         .withColumn("tipo_sesion", F.lit("RACE"))
         .drop("nombre_carrera", "fecha_carrera", "ingesta_timestamp", "archivo_fuente")
)

In [0]:
sprints_df = (
    spark.table(f"{catalogo}.{esquema_silver}.sprints")
         .withColumn("tipo_sesion", F.lit("SPRINT"))
         .drop("nombre_carrera", "fecha_carrera", "ingesta_timestamp", "archivo_fuente")
)

#### 2 - UNION `resultados` y `sprints`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

#### 3 - Agregar columnas adcionales
1. is_win -> Indica que el piloto ganó la carrera
1. is_podium -> Indica que el piloto llegó en las  posicones  (1, 2, 3)
1. has_points -> Indica que el piloto gano puntos


In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("posicion_final") == 1)
        .withColumn("is_podium", F.col("posicion_final").between(1, 3))
        .withColumn("has_points", F.col("puntos") > 0)
)

In [0]:
display(fact_session_results_df.filter("temporada = 2025"))

#### 4 - Escribir la data transformada a la tabla gold `fact_resultado_sesion`

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))